# 34. Exact conditional null for the budget-preserving regime-matched null

In section 4 of notebook 32, the nominal-5% budget-preserving regime-matched null (`draw_budget` of notebooks 26, 31 and 32) rejects 44 of 600 surrogate series that have no predictive skill (7.3%). Here I compute the exact conditional null distribution on the same development period (2023-01 to 2026-04) and use it to check that null.

- Null Q_ν: a uniform draw of the alert set from all placements that respect the alert counts ν in each (month, realized-volatility decile) cell and the 240-minute cooldown.
- Statistic T: the number of pre-onset catches (`pre` from `fast_counts` in notebook 32). The judging window (30–240 minutes before the breach, 210 minutes wide) is shorter than the cooldown and windows do not overlap, so T equals the number of alerts on pre-onset minutes. I compute its distribution under Q_ν exactly by dynamic programming, not by Monte Carlo.
- p-values: I compute the exact p (tail probability, no Monte Carlo error), the randomized exact p̃ (ties broken with a uniform random number, so that p̃ is exactly uniform under Q) and a valid Monte Carlo p from 200 draws of the exact distribution, and set them against the Monte Carlo p of the notebook 26 resampler.
- Sections: (1) validation of the exact distribution and sampler against full enumeration on small designs and on the real design; (2) the real M0 alerts: exact mean and p-value against the resampler null; (3) Q row: I apply the notebook 26 resampler, as is and without the exclusion around original positions, to N_Q = 500 placements that I draw exactly from Q_ν, which separates the effects of the exclusion, of sequential placement with substitute deciles and of Monte Carlo resolution; (4) R row: the exact test on the surrogates of notebook 32 in a fixed order, which reached its run-time limit after the first 214 of the 600, compared pair by pair with notebook 32's p-values; (5) summary.

I fix the reading rules in `34_SPEC.json`, including an implementation check (I require the 99% interval of the rejection rate of p̃ in the Q row to contain α at each of α = 0.01, 0.05 and 0.10), and I report the same tables whatever the results. This analysis is exploratory; it uses only the development period and the stored results of notebooks 16 and 32, and it leaves the preregistered confirmatory plan unchanged. I write the outputs to `exact_conditional_null_v1`. Some cells of this saved copy have no output; `EXACT_NULL_SUMMARY.json` and the tables in that folder hold all results.

In [13]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook 01 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- chunked_hs, as in notebook 01 ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT computation {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook 01 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- price_only_catalog, as in notebook 07 ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df

# Shared with notebook 16 (identical code): VPIN, causal feature set, GBM settings.
REGIME_WINDOW_D = 30; RV_WINDOW = 1440; VPIN_BUCKETS_PER_DAY = 50; VPIN_WINDOW = 50; VPIN_LOOKBACK_D = 30
GBM_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=200, l2_regularization=1.0,
                  class_weight='balanced', early_stopping=False, random_state=17)
FEATS = ['z_vol', 'tsr', 'z_hs', 'ret_5', 'ret_15', 'ret_60', 'ret_240', 'rv_15', 'rv_60', 'rv_240', 'rv_ratio_15_240',
         'dd_240', 'dd_1440', 'vol_intensity_15', 'sell_ratio_15', 'sell_ratio_60', 'vpin']

@njit(cache=False)
def _vpin_core(vol, buy, bucket, n_win):
    out = np.full(len(vol), np.nan); ring = np.zeros(n_win); k = 0; run = 0.0; cv = 0.0; cb = 0.0
    for i in range(len(vol)):
        v = vol[i]; b = buy[i]
        if not (v == v) or not (b == b) or v < 0:
            v = 0.0; b = 0.0
        cv += v; cb += min(max(b, 0.0), v)
        bs = bucket[i]
        if bs == bs and bs > 0 and cv >= bs:
            imb = abs(2.0 * cb - cv) / cv
            j = k % n_win; run += imb - ring[j]; ring[j] = imb; k += 1; cv = 0.0; cb = 0.0
        if k >= n_win:
            out[i] = run / n_win
    return out


def vpin_series(px):
    vol = px.volume.to_numpy(float); buy = px.taker_buy_vol.to_numpy(float)
    daily = px.volume.groupby(px.index.floor('D')).sum()
    size = (daily.rolling(VPIN_LOOKBACK_D, min_periods=VPIN_LOOKBACK_D).mean().shift(1) / VPIN_BUCKETS_PER_DAY)
    bucket = size.reindex(px.index.floor('D')).to_numpy(float)
    return _vpin_core(vol, buy, bucket, VPIN_WINDOW)


def causal_features(px, sg):
    c = px.close.astype(float); lc = np.log(c); r = lc.diff(); vol = px.volume.astype(float); sell = vol - px.taker_buy_vol.astype(float)
    F = pd.DataFrame(index=px.index)
    F['z_vol'] = sg.z_vol.astype(float); F['tsr'] = sg.tsr.astype(float); F['z_hs'] = sg.z_hs.astype(float)
    for k in (5, 15, 60, 240):
        F[f'ret_{k}'] = lc - lc.shift(k)
    for k in (15, 60, 240):
        F[f'rv_{k}'] = r.rolling(k, min_periods=k // 2).std()
    F['rv_ratio_15_240'] = F.rv_15 / F.rv_240
    F['dd_240'] = 1 - c / c.rolling(240, min_periods=120).max()
    F['dd_1440'] = 1 - c / c.rolling(1440, min_periods=720).max()
    F['vol_intensity_15'] = vol.rolling(15, min_periods=8).sum() / (vol.rolling(1440, min_periods=720).sum() / 96)
    F['sell_ratio_15'] = sell.rolling(15, min_periods=8).sum() / vol.rolling(15, min_periods=8).sum()
    F['sell_ratio_60'] = sell.rolling(60, min_periods=30).sum() / vol.rolling(60, min_periods=30).sum()
    F['vpin'] = vpin_series(px)
    return F.replace([np.inf, -np.inf], np.nan).astype('float32')
import bisect
from scipy.stats import chisquare, kstest

OUT = SPOT_BASE / 'exact_conditional_null_v1'; OUT.mkdir(parents=True, exist_ok=True)
M16 = SPOT_BASE / 'methods_definitions_v1'
M32 = SPOT_BASE / 'robustness_checks_v1'
DNAME, DCL = '4pct', 0.04; H_REF = 240
SEED = 20261102; SEED32 = 20261031                    # SEED32: notebook 32 (surrogates are regenerated with its seeds)
N_Q = 500; B_OLD = 200; B_M0_OLD = 1000; ALPHAS = (0.01, 0.05, 0.10)
R_SHIFT = 400; R_BLOCK = 200; SHIFT_MIN_D = 30; N_SHUFFLE_BLOCKS = 40; BP_TRIES = 64   # as in notebook 32
R_MIN_REPORT = 200; MAX_HOURS = 10.0
EVAL_LAST = ts(EVAL_END).value - (LEAD_MAX + 5) * NS
C_MIN = COOLDOWN                                       # cooldown in minutes (240)

for _p in [M16 / 'METHODS_FROZEN.json', M16 / 'METHODS_SPEC.json', M16 / f'alarms_M0_VF_logit_{DNAME}.csv', M32 / '32_SPEC.json', M32 / '4b_null_calibration_pvalues.csv']:
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 16 and 32 are required: {_p}')
FROZEN16 = json.loads((M16 / 'METHODS_FROZEN.json').read_text()); SPEC16 = json.loads((M16 / 'METHODS_SPEC.json').read_text())
if sha256_file(M16 / 'METHODS_SPEC.json') != FROZEN16['spec_sha256']:
    raise RuntimeError('GUARD FAIL: notebook-16 spec differs')
CP32 = pd.read_csv(M32 / '4b_null_calibration_pvalues.csv')
if len(CP32) != R_SHIFT + R_BLOCK:
    raise RuntimeError('GUARD FAIL: notebook-32 surrogate table has an unexpected length')

SPEC = {'title': 'Exact conditional null for the budget-preserving regime-matched alert null (exploratory; appendix)',
        'null_model': 'Q_nu: alert set uniform over all sets on the notebook-32 placement grid with the observed counts per (calendar month, RV decile) cell and spacing >= 240 minutes',
        'statistic': 'pre-onset catches T (notebook-32 fast_counts pre); equals the number of alerts on pre-onset minutes because the 210-minute judging windows are shorter than the cooldown and disjoint (checked)',
        'exact_computation': 'backward dynamic programme over the grid with per-month remaining-count states; saturating counter of T; factorial-moment pass for mean and variance; exact sampler by forward sampling from the same tables',
        'pvalues': {'exact': 'P_Q(T >= T_obs), no Monte Carlo error', 'randomized_exact': 'P_Q(T > T_obs) + U P_Q(T = T_obs)', 'valid_mc': f'(1 + #(T* >= T_obs)) / (B + 1), B = {B_OLD} draws from the exact law (binomial representation)',
                    'old': f'notebook-32 draw_budget, (1 + #) / (B + 1) with B = {B_OLD} (Q row) or the stored notebook-32 value (R row, B = 400)'},
        'q_row': {'draws': N_Q, 'old_draws_per_test': B_OLD, 'variants': ['draw_budget as in notebook 32', 'draw_budget without the >= 240-minute exclusion around the original alert']},
        'r_row': {'surrogates': 'notebook-32 families regenerated with its seeds (400 circular shifts, 200 block derangements); guard: alerts and pre-onset catches equal the notebook-32 table',
                  'order': 'interleaved: shift r, block r for r < 200, then shift 200-399', 'max_hours': MAX_HOURS, 'min_report': R_MIN_REPORT},
        'readings': {'implementation': 'randomized exact p in the Q row: rejection rate 99% interval must contain alpha at 0.01, 0.05, 0.10; otherwise stop interpreting',
                     'old_vs_exact': 'paired difference of rejection decisions with 95% interval and one-sided discordance counts; McNemar exact p',
                     'old_vs_nominal': 'Clopper-Pearson 95% interval of the rejection rate: covers alpha -> no evidence of size distortion; below -> conservative; above -> anti-conservative',
                     'separation': 'exclusion = old - no-exclusion; sequential and nearest-decile = no-exclusion - valid MC from the exact law; MC resolution = valid MC - exact (Q row only)',
                     'r_row': 'surrogates are not independent; intervals are descriptive; surrogate no-skill is not the same hypothesis as Q_nu',
                     'reporting': 'the same tables are reported whatever the size or direction of the differences'},
        'alphas': list(ALPHAS), 'seed': SEED, 'notebook32_seed': SEED32, 'notebook16_spec_sha256': FROZEN16['spec_sha256'],
        'holdouts': 'not used', 'prospective_folder': 'not read', 'status': 'exploratory; appendix; no confirmatory plan changed'}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True, default=str) + '\n').encode()
_sp = OUT / '34_SPEC.json'
if _sp.exists() and _sp.read_bytes() != _raw:
    _prior_results = [OUT / x for x in ('2_m0_exact_vs_budget.csv', '3_q_row_pvalues.csv', '4_r_row_pvalues.csv')]
    if any(p.exists() for p in _prior_results):
        raise RuntimeError('34_SPEC changed but earlier results exist; use a new output folder so that analyses do not mix.')
    _sp.rename(OUT / f'34_SPEC.replaced_{pd.Timestamp.now(tz="UTC"):%Y%m%dT%H%M%S}.json')
_sp.write_bytes(_raw); SPEC_SHA = hashlib.sha256(_raw).hexdigest()
print('Output:', OUT, '| spec sha256', SPEC_SHA[:16])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/exact_conditional_null_v1 | spec sha256 9169a5ff8820351b


## 0. Reproduction of M0's scores, alerts and events (same code as section 0 of notebook 32)

In [14]:
t_ = time.time()
px, sg = load_btc_spot()
F = causal_features(px, sg)
T = px.index.asi8; close = px.close.to_numpy(float); volume = px.volume.to_numpy(float)
v_ = sg.z_vol.to_numpy(float); f_ = sg.tsr.to_numpy(float); h_ = sg.z_hs.to_numpy(float)
VALID = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) & np.isfinite(v_) & np.isfinite(f_) & np.isfinite(h_) & (f_ >= 0) & (f_ <= 1))
EVP = VALID & in_period(T, EVAL_START, EVAL_END)          # notebook-16 alert eligibility
_lc = np.log(np.where(np.isfinite(close) & (close > 0), close, np.nan))
RV = pd.Series(_lc).diff().rolling(RV_WINDOW, min_periods=RV_WINDOW // 2).std().to_numpy()
TCOLS = ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']
E_ref = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')
CAT = price_only_catalog(close, volume, px.index, H=H_REF, decline=DCL)
for c_ in TCOLS:
    CAT[c_] = pd.to_datetime(CAT[c_], utc=True)
CAT = CAT.sort_values('breach_utc').reset_index(drop=True)
if len(CAT) != len(E_ref) or any(not np.array_equal(CAT[c_].astype('int64').to_numpy(), E_ref[c_].astype('int64').to_numpy()) for c_ in TCOLS):
    raise RuntimeError('GUARD FAIL: H=240 catalog differs from the notebook-2 catalog')
EV = events_between(CAT, EVAL_START, EVAL_END).reset_index(drop=True)
Xl = F[['z_vol', 'tsr']].to_numpy(float); okl = train_mask(T, VALID) & np.isfinite(Xl).all(axis=1)
cfg0 = FROZEN16['configs'][f'M0_VF_logit|{DNAME}']
m0 = new_logit().fit(Xl[okl], label_from_catalog(T, CAT)[okl])
if not (np.allclose(m0[-1].coef_.ravel(), cfg0['coef'], rtol=1e-6, atol=1e-9) and np.isclose(m0[-1].intercept_[0], cfg0['intercept'], rtol=1e-6, atol=1e-9)):
    raise RuntimeError('GUARD FAIL: M0 refit differs from notebook 16')
TH0 = float(cfg0['threshold_2022'])
S0_FULL = np.full(len(T), np.nan); okf = EVP & np.isfinite(Xl).all(axis=1); S0_FULL[okf] = m0.predict_proba(Xl[okf])[:, 1]
A_M0 = np.sort(pd.to_datetime(pd.read_csv(M16 / f'alarms_M0_VF_logit_{DNAME}.csv').alarm_utc, utc=True).astype('int64').to_numpy())
if len(A_M0) != cfg0['dev_alerts']:
    raise RuntimeError('GUARD FAIL: notebook-16 M0 alarms changed')
_ok0 = EVP & np.isfinite(S0_FULL)
if not np.array_equal(emit_calendar_fast(T[_ok0], S0_FULL[_ok0] >= TH0), A_M0):
    raise RuntimeError('GUARD FAIL: re-running the frozen scheduler on the M0 score does not reproduce the notebook-16 alerts')
del Xl
print(f'PASS: M0 model, alerts and scheduler verified; {len(EV)} events in the development period ({time.time() - t_:.0f}s)')

# --- functions of notebook 32, unchanged ---
def ev_arrays(ev):
    return dict(br=ev.breach_utc.astype('int64').to_numpy(), on=ev.onset_1pct_utc.astype('int64').to_numpy(), pk=ev.peak_utc.astype('int64').to_numpy(), n=len(ev))


def fast_counts(a, ea):
    a = np.sort(np.asarray(a, np.int64)); used = np.zeros(len(a), bool); pre = post = 0
    for k in range(ea['n']):
        b = ea['br'][k]; l_ = np.searchsorted(a, b - LEAD_MAX * NS, 'left'); r_ = np.searchsorted(a, b - LEAD_MIN * NS, 'right')
        for j in range(l_, r_):
            if not used[j]:
                used[j] = True
                if a[j] < ea['on'][k]:
                    pre += 1
                else:
                    post += 1
                break
    return dict(pre=pre, post=post)


DQ_ORDER = [sorted(range(10), key=lambda d, q=q: (abs(d - q), d)) for q in range(10)]


def make_space(Tp, RVp, start, end):
    okd = (Tp >= start.value) & (Tp < end.value) & np.isfinite(RVp)
    tt = Tp[okd]; dq = np.minimum((pd.Series(RVp[okd]).rank(pct=True).to_numpy() * 10).astype(int), 9)
    mo = cal_fields(tt)[1]; key = mo * 10 + dq; o2 = np.lexsort((tt, key))
    return dict(TT=tt, DD=dq, PK=tt[o2], KS=key[o2])


def draw_budget(a, rng, sp, audit=None, exclude_original=True):
    """Budget-preserving regime-matched draw (notebook 26/31/32 code): same calendar month, same (or nearest) RV decile, >= 240 min from the original
    and from every alert already placed; if the decile search fails, every minute of the month is scanned; only if no feasible minute exists does the alert keep its time.
    exclude_original=False (notebook 34 only) drops the >= 240-minute condition relative to the alert's own original time; nothing else changes."""
    a = np.asarray(a, np.int64); j = np.clip(np.searchsorted(sp['TT'], a, 'right') - 1, 0, len(sp['TT']) - 1)
    q0 = sp['DD'][j]; mo = cal_fields(a)[1]; KS, PK = sp['KS'], sp['PK']
    placed = []; out = np.empty(len(a), np.int64); exact = near = anyq = failed = viol = 0
    for i in rng.permutation(len(a)):
        ai = int(a[i]); done = False
        for dq in DQ_ORDER[q0[i]]:
            key = mo[i] * 10 + dq; lo = np.searchsorted(KS, key, 'left'); hi = np.searchsorted(KS, key, 'right')
            if hi <= lo:
                continue
            for c in PK[lo + rng.integers(0, hi - lo, BP_TRIES)]:
                c = int(c)
                if exclude_original and abs(c - ai) < LEAD_MAX * NS:
                    continue
                k = bisect.bisect_left(placed, c)
                if (k > 0 and c - placed[k - 1] < COOLDOWN * NS) or (k < len(placed) and placed[k] - c < COOLDOWN * NS):
                    continue
                placed.insert(k, c); out[i] = c; done = True; break
            if done:
                exact += int(dq == q0[i]); near += int(dq != q0[i]); break
        if not done:
            lo = np.searchsorted(KS, mo[i] * 10, 'left'); hi = np.searchsorted(KS, mo[i] * 10 + 9, 'right'); cand = PK[lo:hi]
            if exclude_original:
                cand = cand[np.abs(cand - ai) >= LEAD_MAX * NS]
            if len(placed) and len(cand):
                pl = np.asarray(placed); kk = np.searchsorted(pl, cand)
                left_ok = np.where(kk > 0, cand - pl[np.maximum(kk - 1, 0)] >= COOLDOWN * NS, True)
                right_ok = np.where(kk < len(pl), pl[np.minimum(kk, len(pl) - 1)] - cand >= COOLDOWN * NS, True)
                cand = cand[left_ok & right_ok]
            if len(cand):
                c = int(cand[rng.integers(0, len(cand))]); bisect.insort(placed, c); out[i] = c; anyq += 1
            else:
                failed += 1; out[i] = ai
                k = bisect.bisect_left(placed, ai)
                if (k > 0 and ai - placed[k - 1] < COOLDOWN * NS) or (k < len(placed) and placed[k] - ai < COOLDOWN * NS):
                    viol += 1
                placed.insert(k, ai)
    if audit is not None:
        audit['exact'] += exact; audit['near'] += near; audit['any'] += anyq; audit['failed'] += failed; audit['viol'] += viol; audit['n'] += len(a)
    return np.sort(out)


EA4 = ev_arrays(EV); SP = make_space(T, np.where(VALID, RV, np.nan), ts(EVAL_START), ts(EVAL_END))
mm0 = match_events(EV, A_M0); fc0 = fast_counts(A_M0, EA4)
if (int(mm0.pre1pct.sum()), int((~mm0.pre1pct).sum())) != (fc0['pre'], fc0['post']):
    raise RuntimeError('GUARD FAIL: fast matcher differs from match_events')
print('M0 pre-onset catches:', fc0['pre'], '| grid minutes:', len(SP['TT']))

PASS: M0 model, alerts and scheduler verified; 83 events in the development period (21s)
M0 pre-onset catches: 22 | grid minutes: 1750857


## 1. Exact distribution and exact sampler: implementation and validation

My dynamic program counts backward over a time-ordered grid, with the alerts still to place in each cell as the state; placing an alert moves the next position to the first grid point 240 minutes later, and the cooldown applies across month boundaries. To validate it, I compare it with full enumeration on several hundred small random designs and check the sampler's frequencies with a chi-square test.

In [15]:
"""Exact conditional null for budget-constrained alerts (notebook 34 core).
Q_nu: uniform over alert sets A ⊂ positions with spacing >= C (time units) and exact counts nu[month, class].
Positions are sorted times; each has a month index, a class, an indicator h (pre-onset minute) and a weight w (1 = uniform).
Statistic T(A) = sum_{t in A} h(t) (additive because the judging window is shorter than the cooldown; checked separately on the real data)."""
import numpy as np
from numba import njit

# ---------------------------------------------------------------- design preparation
def prepare(times, month, cls, h, nu, C, w=None):
    """Drop positions whose (month, class) has zero count (they can never hold an alert), build radix states per month.
    month: int array (any labels, increasing in time); cls: int in [0, K); nu: dict {(month_label, class): count}."""
    times = np.asarray(times, np.int64); month = np.asarray(month, np.int64); cls = np.asarray(cls, np.int64); h = np.asarray(h, np.int64)
    assert np.all(np.diff(times) > 0)
    w = np.ones(len(times)) if w is None else np.asarray(w, float)
    K = int(max(cls.max() + 1, max(b for _, b in nu) + 1)) if nu else int(cls.max() + 1)
    mlabels = sorted({m for (m, b), v in nu.items() if v > 0})
    mid = {m: k for k, m in enumerate(mlabels)}; M = len(mlabels)
    NU = np.zeros((M, K), np.int64)
    for (m, b), v in nu.items():
        if v > 0: NU[mid[m], b] = v
    ml = np.array(mlabels, np.int64); pos = np.searchsorted(ml, month); pos_c = np.clip(pos, 0, max(M - 1, 0))
    keep = (M > 0) & (pos < M) & (ml[pos_c] == month) if M else np.zeros(len(month), bool)
    if M: keep &= NU[pos_c, np.clip(cls, 0, K - 1)] > 0
    t = times[keep]; mo = pos_c[keep].astype(np.int64); c = cls[keep]; hh = h[keep]; ww = w[keep]
    if M and not np.all(np.diff(mo) >= 0): raise ValueError('months must be increasing in time')
    L = len(t)
    mstart = np.searchsorted(mo, np.arange(M), 'left'); mend = np.searchsorted(mo, np.arange(M), 'right')
    if np.any(mend <= mstart): raise ValueError('a month with a positive count has no admissible position')
    RAD = NU + 1; STR = np.zeros((M, K), np.int64); S = np.ones(M, np.int64); FULL = np.zeros(M, np.int64)
    for m in range(M):
        s = 1
        for b in range(K):
            if NU[m, b] > 0: STR[m, b] = s; s *= RAD[m, b]
        S[m] = s; FULL[m] = int((NU[m] * STR[m]).sum())
    st = STR[mo, c]; rad = RAD[mo, c]
    nx = np.searchsorted(t, t + C, 'left').astype(np.int64)
    R = int((nx - np.arange(L)).max()) + 1 if L else 1
    return dict(t=t, mon=mo, cls=c, h=hh, w=ww, st=st, rad=rad, nx=nx, S=S, FULL=FULL, mstart=mstart, mend=mend, M=M, L=L, R=R,
                NU=NU, keep=keep, mlabels=mlabels, n=int(NU.sum()))

# ---------------------------------------------------------------- backward pass
@njit(cache=False)
def _pairs(D, mode, hi):
    """(destination bin, source bin) pairs for multiplying by u^hi (mode 0, saturating) or (1 + x)^hi (mode 1, truncated)."""
    if hi == 0:
        P = np.empty((D, 2), np.int64)
        for k in range(D): P[k, 0] = k; P[k, 1] = k
        return P
    if mode == 0:
        P = np.empty((D, 2), np.int64)
        for k in range(D - 1): P[k, 0] = k + 1; P[k, 1] = k
        P[D - 1, 0] = D - 1; P[D - 1, 1] = D - 1
        return P
    P = np.empty((2 * D - 1, 2), np.int64)
    for k in range(D): P[k, 0] = k; P[k, 1] = k
    for k in range(D - 1): P[D + k, 0] = k + 1; P[D + k, 1] = k
    return P

@njit(cache=False)
def _backward(nx, mon, st, rad, h, w, S, FULL, D, mode, R):
    """Backward recursion F(i, r; u) over kept positions. mode 0: saturating counter of T (bins 0..D-2 exact, bin D-1 = '>= D-1');
    mode 1: truncated polynomial in x = u - 1 (coefficients Z, Z E[T], Z E[T(T-1)]/2, ...). Ring values share one running log scale e;
    Gfull[i] is the value at the full state of month(i) with its scale lcfull[i] (true value = Gfull * exp(lcfull))."""
    L = len(nx); M = len(S); Smax = 1
    for m in range(M):
        if S[m] > Smax: Smax = S[m]
    ring = np.zeros((R, D, Smax)); Gfull = np.zeros((L, D)); lcfull = np.full(L, -np.inf)
    P0 = _pairs(D, mode, 0); P1 = _pairs(D, mode, 1); e = 0.0; BIG = 1e200; LBIG = np.log(BIG)
    for i in range(L - 1, -1, -1):
        m = mon[i]; Sm = S[m]; cur = ring[i % R]
        # skip term
        if i + 1 < L and mon[i + 1] == m:
            src = ring[(i + 1) % R]
            for k in range(D):
                for s in range(Sm): cur[k, s] = src[k, s]
        else:
            for k in range(D):
                for s in range(Sm): cur[k, s] = 0.0
            if i + 1 < L:
                if lcfull[i + 1] > -np.inf:
                    f = np.exp(lcfull[i + 1] - e)
                    for k in range(D): cur[k, 0] = f * Gfull[i + 1, k]
            else:
                cur[0, 0] = np.exp(-e)
        # take term
        j = nx[i]; wi = w[i]; stb = st[i]; rb = rad[i]
        PP = P1 if h[i] == 1 else P0
        if j < L and mon[j] == m:
            src = ring[j % R]; block = stb * rb; nq = Sm // block
            for p in range(PP.shape[0]):
                kd = PP[p, 0]; ks = PP[p, 1]
                for q in range(nq):
                    base = q * block
                    for r in range(1, rb):
                        o = base + r * stb
                        for l in range(stb):
                            cur[kd, o + l] += wi * src[ks, o - stb + l]
        elif stb < Sm:
            ok = False; f = 0.0
            if j < L and mon[j] == m + 1 and lcfull[j] > -np.inf:
                ok = True; f = wi * np.exp(lcfull[j] - e)
            elif j >= L and m == M - 1:
                ok = True; f = wi * np.exp(-e)
            if ok:
                for p in range(PP.shape[0]):
                    kd = PP[p, 0]; ks = PP[p, 1]
                    if j < L: cur[kd, stb] += f * Gfull[j, ks]
                    elif ks == 0: cur[kd, stb] += f
        # occasional rescale of the whole ring (values only grow as i decreases)
        if (i & 15) == 0 or (i + 1 < L and mon[i + 1] != m):
            mx = 0.0
            for k in range(D):
                for s in range(Sm):
                    if cur[k, s] > mx: mx = cur[k, s]
            if mx > BIG:
                ring *= 1.0 / BIG; e += LBIG
        f_ = FULL[m]
        for k in range(D): Gfull[i, k] = cur[k, f_]
        lcfull[i] = e
    return Gfull, lcfull

def exact_law(P, kcap):
    """Exact law of T under Q_nu: P(T = k) for k < kcap and P(T >= kcap) (last entry)."""
    G, lc = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], int(kcap) + 1, 0, P['R'])
    v = G[0]
    if not v.sum() > 0: raise ValueError('no admissible configuration: the counts cannot be placed under the cooldown')
    return v / v.sum(), lc[0] + np.log(v.sum())

def exact_moments(P):
    """Exact E[T] and Var[T] under Q_nu from the factorial-moment pass."""
    G, lc = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], 3, 1, P['R'])
    z, f1, f2 = G[0]; mean = f1 / z; e2 = 2 * f2 / z           # E[T(T-1)] = 2 * coefficient of x^2
    return mean, e2 + mean - mean ** 2

def log_count(P):
    """log of the number (weighted) of admissible configurations."""
    G, lc = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], 1, 0, P['R'])
    return lc[0] + np.log(G[0, 0])

# ---------------------------------------------------------------- exact sampler
@njit(cache=False)
def _rows(m, lo, hi, after, after_e, nx, mon, st, rad, w, S, mend, Gfull, lcfull):
    """Backward rows (D = 1, u = 1) of month m for positions [lo, hi), given the rows of [hi, hi + len(after)) (empty when hi is the month end).
    Returns rows for [lo, hi + len(after)); row r holds F(lo + r, .) * exp(-erow[r])."""
    b = mend[m]; Sm = S[m]; L = len(nx); M = len(S); nA = after.shape[0]; tot = hi - lo + nA
    T = np.zeros((tot, Sm)); erow = np.zeros(tot); BIG = 1e200; LBIG = np.log(BIG)
    for r in range(nA):
        erow[hi - lo + r] = after_e[r]
        for s in range(Sm): T[hi - lo + r, s] = after[r, s]
    if nA > 0: e = after_e[0]
    else:
        e = lcfull[b] if b < L else 0.0
        if e == -np.inf: e = 0.0
    for i in range(hi - 1, lo - 1, -1):
        il = i - lo
        if i + 1 < b:
            f = np.exp(erow[il + 1] - e)
            for s in range(Sm): T[il, s] = f * T[il + 1, s]
        elif i + 1 < L:
            if lcfull[i + 1] > -np.inf: T[il, 0] = np.exp(lcfull[i + 1] - e) * Gfull[i + 1, 0]
        else:
            T[il, 0] = np.exp(-e)
        j = nx[i]; stb = st[i]; rb = rad[i]; wi = w[i]
        if j < b:
            f = wi * np.exp(erow[j - lo] - e); block = stb * rb; nq = Sm // block; jl = j - lo
            for q in range(nq):
                base = q * block
                for r in range(1, rb):
                    o = base + r * stb
                    for l in range(stb):
                        T[il, o + l] += f * T[jl, o - stb + l]
        elif stb < Sm:
            if j < L and mon[j] == m + 1 and lcfull[j] > -np.inf: T[il, stb] += wi * np.exp(lcfull[j] - e) * Gfull[j, 0]
            elif j >= L and m == M - 1: T[il, stb] += wi * np.exp(-e)
        mx = 0.0
        for s in range(Sm):
            if T[il, s] > mx: mx = T[il, s]
        if mx > BIG:
            for s in range(Sm): T[il, s] /= BIG
            e += LBIG
        erow[il] = e
    return T, erow

@njit(cache=False)
def _walk_seg(m, lo, hi, T, erow, nx, mon, st, rad, w, mend, FULL, Gfull, lcfull, cur_i, cur_s, out, nout, seed):
    """Advance every draw whose current position lies in [lo, hi) of month m until it leaves the segment."""
    np.random.seed(seed)
    b = mend[m]; L = len(nx); M = len(FULL)
    for d in range(len(cur_i)):
        i = cur_i[d]; s = cur_s[d]
        if i < lo or i >= hi: continue
        while i < hi:
            if s == 0:
                i = b; break
            il = i - lo; j = nx[i]; tv = 0.0; stb = st[i]
            if (s // stb) % rad[i] >= 1:
                if j < b:
                    tv = w[i] * T[j - lo, s - stb] * np.exp(erow[j - lo] - erow[il])
                elif s - stb == 0:
                    if j < L and mon[j] == m + 1: tv = w[i] * Gfull[j, 0] * np.exp(lcfull[j] - erow[il])
                    elif j >= L and m == M - 1: tv = w[i] * np.exp(-erow[il])
            sv = T[il + 1, s] * np.exp(erow[il + 1] - erow[il]) if i + 1 < b else 0.0
            p = tv / (tv + sv) if tv + sv > 0.0 else 0.0
            if np.random.random() < p:
                out[d, nout[d]] = i; nout[d] += 1; s -= stb; i = j
                if j >= b: break
            else:
                i += 1
        cur_i[d] = i
        if i >= b: cur_s[d] = FULL[mon[i]] if i < L else 0
        else: cur_s[d] = s

def sample_exact(P, ndraw, seed=0, seg=4000):
    """ndraw independent exact draws from Q_nu; returns an (ndraw, n) array of kept-position indices (sorted per draw).
    Each month is processed in segments of `seg` positions: one backward pass stores only the rows just after each segment boundary,
    and each segment is recomputed from its checkpoint when the draws walk through it (memory about (seg + R) x states per month)."""
    G, lcf = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], 1, 0, P['R'])
    n = P['n']; R = P['R']; out = np.full((ndraw, n), -1, np.int64); nout = np.zeros(ndraw, np.int64)
    cur_i = np.zeros(ndraw, np.int64); cur_s = np.full(ndraw, P['FULL'][0], np.int64)
    args = (P['nx'], P['mon'], P['st'], P['rad'], P['w'], P['S'], P['mend'], G, lcf)
    for m in range(P['M']):
        a, b = int(P['mstart'][m]), int(P['mend'][m]); Sm = int(P['S'][m])
        bounds = list(range(a, b, seg)) + [b]; nseg = len(bounds) - 1
        ck = [None] * nseg; after = np.zeros((0, Sm)); after_e = np.zeros(0)
        for k in range(nseg - 1, -1, -1):
            lo, hi = bounds[k], bounds[k + 1]
            ck[k] = (after, after_e)
            Tm, er = _rows(m, lo, hi, after, after_e, *args)
            nA = min(R, b - lo); after = Tm[:nA].copy(); after_e = er[:nA].copy()
        for k in range(nseg):
            lo, hi = bounds[k], bounds[k + 1]
            Tm, er = _rows(m, lo, hi, ck[k][0], ck[k][1], *args)
            _walk_seg(m, lo, hi, Tm, er, P['nx'], P['mon'], P['st'], P['rad'], P['w'], P['mend'], P['FULL'], G, lcf,
                      cur_i, cur_s, out, nout, (seed * 1000003 + m) * 1009 + k)
    if np.any(nout != n): raise RuntimeError('sampler did not place every alert')
    return out

def check_draws(P, idx, C):
    """Constraint audit of exact draws: counts per (month, class) and spacing."""
    t = P['t'][idx]; bad_space = int((np.diff(t, axis=1) < C).sum())
    bad_count = 0
    for d in range(idx.shape[0]):
        cnt = np.zeros_like(P['NU']); np.add.at(cnt, (P['mon'][idx[d]], P['cls'][idx[d]]), 1)
        bad_count += int(np.any(cnt != P['NU']))
    return bad_space, bad_count
import itertools, random


def _enumerate(times, month, cls, nu, C):
    n = sum(nu.values()); out = []
    for comb in itertools.combinations(range(len(times)), n):
        if any(times[comb[k + 1]] - times[comb[k]] < C for k in range(n - 1)):
            continue
        cnt = {}
        for i in comb:
            cnt[(month[i], cls[i])] = cnt.get((month[i], cls[i]), 0) + 1
        if all(cnt.get(c, 0) == v for c, v in nu.items() if v > 0) and all(nu.get(c, 0) == v for c, v in cnt.items()):
            out.append(comb)
    return out


random.seed(SEED); _r = np.random.default_rng(SEED)
vrows = []; chi_p = []; t_ = time.time()
for trial in range(1500):
    L_ = random.randint(6, 13); times = np.cumsum([random.choice([1, 1, 1, 2, 3]) for _ in range(L_)])
    M_ = random.randint(1, 4); K_ = random.randint(1, 3); C_ = random.randint(1, 7)
    month = np.sort(_r.integers(0, M_, L_)) * 7 + 3; cls = _r.integers(0, K_, L_); h = (_r.random(L_) < 0.35).astype(int)
    w = np.where(_r.random(L_) < 0.5, 1.0, _r.choice([0.5, 2.0, 3.0], L_)) if trial % 3 == 0 else np.ones(L_)
    nu = {c: random.randint(0, 2) for c in sorted(set(zip(month.tolist(), cls.tolist())))}
    if sum(nu.values()) == 0:
        continue
    try:
        P_ = prepare(times, month, cls, h, nu, C_, w)
    except ValueError:
        continue
    A_ = _enumerate(times.tolist(), month.tolist(), cls.tolist(), nu, C_)
    if not A_:
        continue
    wt = np.array([np.prod(w[list(a)]) for a in A_]); Tv = np.array([h[list(a)].sum() for a in A_]); kmax = int(Tv.max()) + 2
    ref = np.array([wt[Tv == k].sum() for k in range(kmax)]); ref = np.r_[ref[:kmax - 1], wt[Tv >= kmax - 1].sum()] / wt.sum()
    law_, _ = exact_law(P_, kmax - 1); mu_, var_ = exact_moments(P_)
    mref = (wt * Tv).sum() / wt.sum(); vref = (wt * Tv ** 2).sum() / wt.sum() - mref ** 2
    row = dict(trial=trial, configs=len(A_), law_err=float(np.abs(law_ - ref).max()), mean_err=abs(mu_ - mref), var_err=abs(var_ - vref),
               logcount_err=abs(log_count(P_) - np.log(wt.sum())))
    if trial % 4 == 0 and len(A_) > 1:
        idx_ = sample_exact(P_, 8000, seed=trial); bs, bc = check_draws(P_, idx_, C_)
        key = {a: k for k, a in enumerate(A_)}; cnt = np.zeros(len(A_))
        for r_ in np.flatnonzero(P_['keep'])[idx_]:
            cnt[key[tuple(sorted(r_.tolist()))]] += 1
        row.update(sampler_violations=bs + bc, chi_p=float(chisquare(cnt, 8000 * wt / wt.sum()).pvalue))
    vrows.append(row)
VR = pd.DataFrame(vrows); VR.to_csv(OUT / '1_verification_small.csv', index=False)
n_samp = int(VR.chi_p.notna().sum()); share_low = float((VR.chi_p < 0.01).mean()) if n_samp else np.nan
ok_small = (VR.law_err.max() < 1e-12) and (VR.mean_err.max() < 1e-9) and (VR.var_err.max() < 1e-8) and (VR.logcount_err.max() < 1e-9) and (VR.sampler_violations.fillna(0).max() == 0)
ok_chi = binomtest(int((VR.chi_p < 0.01).sum()), n_samp, 0.01).pvalue > 0.001
print(f'{len(VR)} small designs: max law error {VR.law_err.max():.1e}, max mean error {VR.mean_err.max():.1e}; sampler: {n_samp} designs, '
      f'violations {int(VR.sampler_violations.fillna(0).max())}, share of chi-square p < 0.01 = {share_low:.3f} ({time.time() - t_:.0f}s)')
if not (ok_small and ok_chi):
    raise RuntimeError('Exact computation or sampler failed the small-problem verification')
print('PASS: exact law, moments and sampler agree with full enumeration')

567 small designs: max law error 0.0e+00, max mean error 0.0e+00; sampler: 118 designs, violations 0, share of chi-square p < 0.01 = 0.002 (22s)
PASS: exact law, moments and sampler agree with full enumeration


## 2. Real design and M0: exact mean and exact p-value against the resampler null

I take the grid and the (month, decile) cells from `make_space` and `draw_budget` of notebook 32 (valid development minutes; deciles are RV ranks among those minutes). A minute is pre-onset, h(t) = 1, when it lies inside some event's judging window (240–30 minutes before the breach) and before that event's onset.

In [ ]:
TT = SP['TT']; GRID_MIN = (TT // NS).astype(np.int64); GRID_MON = cal_fields(TT)[1]; GRID_DEC = SP['DD'].astype(np.int64)
lo_w = EA4['br'] - LEAD_MAX * NS; hi_w = EA4['br'] - LEAD_MIN * NS
if np.any(lo_w[1:] <= hi_w[:-1]):
    raise RuntimeError('GUARD FAIL: judging windows overlap, so T is not additive')
if (LEAD_MAX - LEAD_MIN) >= COOLDOWN:
    raise RuntimeError('GUARD FAIL: window width is not shorter than the cooldown')
H_GRID = np.zeros(len(TT), np.int64)
for k in range(EA4['n']):
    l_ = np.searchsorted(TT, lo_w[k], 'left'); r_ = np.searchsorted(TT, hi_w[k], 'right')
    H_GRID[l_:r_] = (TT[l_:r_] < EA4['on'][k]).astype(np.int64)
print('pre-onset grid minutes:', int(H_GRID.sum()), '| events with at least one:', int(sum(H_GRID[np.searchsorted(TT, lo_w[k]):np.searchsorted(TT, hi_w[k], 'right')].any() for k in range(EA4['n']))))


def cells_of(a):
    """(month, decile) cell of each alert by the draw_budget rule; also whether the alert minute is itself a grid minute."""
    a = np.asarray(a, np.int64); j = np.clip(np.searchsorted(TT, a, 'right') - 1, 0, len(TT) - 1)
    return cal_fields(a)[1], GRID_DEC[j], TT[j] == a


def design_for(a):
    mo, dq, on_grid = cells_of(a)
    nu = {}
    for m_, d_ in zip(mo.tolist(), dq.tolist()):
        nu[(m_, d_)] = nu.get((m_, d_), 0) + 1
    return prepare(GRID_MIN, GRID_MON, GRID_DEC, H_GRID, nu, C_MIN), int((~on_grid).sum())


def t_additive(a):
    """Sum over alerts of the pre-onset indicator, computed from the event windows directly (not from the grid)."""
    a = np.asarray(a, np.int64); k = np.searchsorted(lo_w, a, 'right') - 1; ok = k >= 0
    kk = np.clip(k, 0, len(lo_w) - 1)
    return int((ok & (a <= hi_w[kk]) & (a < EA4['on'][kk])).sum())


def p_from_law(law, t_obs, u):
    """Exact tail P(T >= t_obs) and the randomized p = P(T > t_obs) + u P(T = t_obs); law has bins 0..K-1 and a last bin '>= K', K > t_obs."""
    K = len(law) - 1
    if t_obs >= K:
        raise ValueError('the law must be computed with an upper bin above t_obs')
    tail = law[t_obs:].sum(); gt = law[t_obs + 1:].sum()
    return float(min(1.0, tail)), float(min(1.0, gt + u * (tail - gt)))


t_ = time.time()
P0, off0 = design_for(A_M0); T0 = fc0['pre']
if off0:
    raise RuntimeError(f'{off0} observed M0 alerts are not on the candidate grid; Q does not contain the observed allocation.')
if t_additive(A_M0) != T0:
    raise RuntimeError('GUARD FAIL: sum of h over M0 alerts differs from fast_counts pre')
KCAP0 = max(T0 + 1, 25)
# Memory check before allocating the large backward-DP ring (coarse memory lower bounds).
_smax = int(P0['S'].max()); _L = int(P0['L']); _R = int(P0['R'])
_gib = 1024.0**3
_ring_gib = _R * (KCAP0 + 1) * _smax * 8 / _gib
_law_output_gib = _L * (KCAP0 + 1) * 8 / _gib
_slab_gib = (4000 + _R) * _smax * 8 / _gib
_checkpoint_gib = sum(int(np.ceil((P0['mend'][i] - P0['mstart'][i]) / 4000)) * min(_R, int(P0['mend'][i]-P0['mstart'][i])) * int(P0['S'][i]) * 8 for i in range(P0['M'])) / _gib
print(f'Memory estimate: max DP states/month={_smax:,}, positions={_L:,}, R={_R}; '
      f'law ring={_ring_gib:.2f} GiB, full-law output={_law_output_gib:.2f} GiB; '
      f'sampler largest slab={_slab_gib:.2f} GiB, checkpoints≈{_checkpoint_gib:.2f} GiB (not peak total RAM).', flush=True)
if max(_ring_gib, _slab_gib, _checkpoint_gib) > 1.0:
    print('Warning: one or more single DP allocations exceed 1 GiB; check Colab RAM before proceeding.', flush=True)
LAW0, _ = exact_law(P0, KCAP0); MU0, VAR0 = exact_moments(P0)
rng_ = np.random.default_rng(SEED + 1)
p0, p0r = p_from_law(LAW0, T0, rng_.random())
print(f'M0 design: {P0["n"]} alerts, {P0["M"]} months, grid positions kept {P0["L"]}, per-month state sizes median {int(np.median(P0["S"]))} / max {int(P0["S"].max())}, '
      f'alerts off the grid {off0} ({time.time() - t_:.0f}s)')
aud0 = dict(exact=0, near=0, any=0, failed=0, viol=0, n=0); rb_ = np.random.default_rng(SEED + 2)
pre_old0 = np.array([fast_counts(draw_budget(A_M0, rb_, SP, aud0), EA4)['pre'] for _ in range(B_M0_OLD)])
p_old0 = (1 + int((pre_old0 >= T0).sum())) / (B_M0_OLD + 1)
T2 = pd.DataFrame([dict(alerts=len(A_M0), events=EA4['n'], pre_observed=T0,
                        exact_mean=MU0, exact_sd=float(np.sqrt(VAR0)), exact_p=p0, exact_p_randomized=p0r, exact_law_upper_bin=KCAP0,
                        old_null_mean=float(pre_old0.mean()), old_null_sd=float(pre_old0.std()), old_p_mc=p_old0, old_draws=B_M0_OLD,
                        old_share_same_decile=aud0['exact'] / aud0['n'], old_share_nearest_decile=aud0['near'] / aud0['n'], old_share_any_minute=aud0['any'] / aud0['n'], old_share_unplaced=aud0['failed'] / aud0['n'])])
T2.to_csv(OUT / '2_m0_exact_vs_budget.csv', index=False); display(T2.T)
pd.DataFrame(dict(k=np.arange(len(LAW0)), prob=LAW0, tail=np.cumsum(LAW0[::-1])[::-1], last_bin_means_at_least=np.arange(len(LAW0)) == len(LAW0) - 1)).to_csv(OUT / '2b_m0_exact_law.csv', index=False)

# exact draws for section 3, and the real-design sampler check (constraints and distribution)
t_ = time.time()
IDX_Q = sample_exact(P0, N_Q, seed=SEED + 3)
bs, bc = check_draws(P0, IDX_Q, C_MIN)
TQ = P0['h'][IDX_Q].sum(1)
cnt = np.bincount(np.minimum(TQ, KCAP0), minlength=KCAP0 + 1); exp_ = LAW0 * N_Q; okb = exp_ >= 5
chi_real = float(chisquare(cnt[okb], exp_[okb] / exp_[okb].sum() * cnt[okb].sum()).pvalue)
A_Q = [TT[np.searchsorted(GRID_MIN, P0['t'][r])] for r in IDX_Q]
add_ok = all(fast_counts(a, EA4)['pre'] == t for a, t in zip(A_Q, TQ))
print(f'{N_Q} exact draws ({time.time() - t_:.0f}s): spacing violations {bs}, count violations {bc}; mean T {TQ.mean():.3f} vs exact {MU0:.3f}; '
      f'chi-square p {chi_real:.3f}; fast_counts equals sum of h in every draw: {add_ok}')
pd.DataFrame([dict(draws=N_Q, spacing_violations=bs, count_violations=bc, mean_T_draws=float(TQ.mean()), exact_mean=MU0, chi_square_p=chi_real, additivity_ok=add_ok)]).to_csv(OUT / '1b_verification_real_design.csv', index=False)
if bs or bc or chi_real <= 0.001 or not add_ok:
    raise RuntimeError('Real-design sampler check failed')
print('PASS: exact draws on the real design satisfy every constraint and follow the exact law')

pre-onset grid minutes: 10268 | events with at least one: 83
Memory estimate: max DP states/month=25,920, positions=1,414,298, R=241; law ring=1.21 GiB, full-law output=0.27 GiB; sampler largest slab=0.82 GiB, checkpoints≈4.40 GiB (not peak total RAM).


## 3. Q row: the size of the notebook 26 resampler under the exact null

I treat each alert placement that I draw exactly from Q_ν (M0's cell counts) as the observed alerts and draw 200 null samples with (a) notebook 32's `draw_budget` as is and (b) a variant that drops only the 240-minute exclusion around original positions. For the same placements I compute the exact p, the randomized exact p and the Monte Carlo p from the exact distribution, and compare them in pairs.

In [ ]:
qpath = OUT / '3_q_row_pvalues.csv'
QR = pd.read_csv(qpath) if qpath.exists() else pd.DataFrame()
done = set(QR.draw) if len(QR) else set(); rows = QR.to_dict('records') if len(QR) else []
t_ = time.time(); n_new = 0
for d in range(N_Q):
    if d in done:
        continue
    a = A_Q[d]; t_obs = int(TQ[d]); rr = np.random.default_rng(SEED + 100000 + d)
    pe, pr = p_from_law(LAW0, t_obs, rr.random())
    p_mc = (1 + rr.binomial(B_OLD, pe)) / (B_OLD + 1)
    row = dict(draw=d, T_obs=t_obs, p_exact=pe, p_exact_randomized=pr, p_valid_mc=p_mc)
    for name, excl in [('old', True), ('old_noexcl', False)]:
        aud = dict(exact=0, near=0, any=0, failed=0, viol=0, n=0); rb = np.random.default_rng(SEED + (200000 if excl else 300000) + d)
        pre = np.array([fast_counts(draw_budget(a, rb, SP, aud, exclude_original=excl), EA4)['pre'] for _ in range(B_OLD)])
        row.update({f'p_{name}': (1 + int((pre >= t_obs).sum())) / (B_OLD + 1), f'{name}_null_mean': float(pre.mean()),
                    f'{name}_share_nearest_decile': aud['near'] / aud['n'], f'{name}_share_any_minute': aud['any'] / aud['n'], f'{name}_share_unplaced': aud['failed'] / aud['n']})
    rows.append(row); n_new += 1
    if n_new % 10 == 0 or d == N_Q - 1:
        pd.DataFrame(rows).to_csv(qpath, index=False)
        el = time.time() - t_; left = N_Q - len(rows)
        print(f'Q row {len(rows)}/{N_Q} ({el / 60:.1f} min; about {el / n_new * left / 60:.0f} min left)', flush=True)
QR = pd.DataFrame(rows).sort_values('draw').reset_index(drop=True); QR.to_csv(qpath, index=False)


def rate_ci(k, n):
    lo, hi = binomtest(int(k), int(n)).proportion_ci(0.95); lo99, hi99 = binomtest(int(k), int(n)).proportion_ci(0.99)
    return float(lo), float(hi), float(lo99), float(hi99)


def paired(x, y):
    n10 = int((x & ~y).sum()); n01 = int((~x & y).sum()); n = len(x); d = (n10 - n01) / n
    se = np.sqrt(max(n10 + n01 - (n10 - n01) ** 2 / n, 0)) / n
    return dict(diff=d, diff_lo=d - 1.96 * se, diff_hi=d + 1.96 * se, only_first=n10, only_second=n01, mcnemar_p=mcnemar_two_sided(n10, n01))


def verdict(lo, hi, a):
    return 'no evidence of size distortion' if lo <= a <= hi else ('conservative' if hi < a else 'anti-conservative')


PCOLS = [('p_exact', 'exact (no MC error)'), ('p_exact_randomized', 'randomized exact'), ('p_valid_mc', f'valid MC from the exact law (B={B_OLD})'),
         ('p_old_noexcl', f'old without own-original exclusion (B={B_OLD})'), ('p_old', f'old draw_budget (B={B_OLD})')]
rrows = []; impl_ok = True
for al in ALPHAS:
    for col, lab in PCOLS:
        k = int((QR[col] <= al).sum()); lo, hi, lo99, hi99 = rate_ci(k, len(QR))
        rrows.append(dict(alpha=al, test=lab, n=len(QR), rejections=k, rate=k / len(QR), lo95=lo, hi95=hi, lo99=lo99, hi99=hi99, vs_nominal=verdict(lo, hi, al)))
        if col == 'p_exact_randomized' and not (lo99 <= al <= hi99):
            impl_ok = False
RQ = pd.DataFrame(rrows); RQ.to_csv(OUT / '3b_q_row_rates.csv', index=False); display(RQ.round(4))
print('implementation check (randomized exact p covers alpha at 99%):', 'PASS' if impl_ok else 'FAIL: the comparisons below are not interpretable')
if not impl_ok:
    raise RuntimeError('Randomized exact p-value failed the pre-specified calibration check')
srows = []
for al in ALPHAS:
    R_ = {c: (QR[c] <= al).to_numpy() for c, _ in PCOLS}
    for a_, b_, what in [('p_old', 'p_exact', 'old vs exact (total, Q row)'), ('p_old', 'p_old_noexcl', 'own-original exclusion'),
                         ('p_old_noexcl', 'p_valid_mc', 'sequential placement and nearest-decile fallback'), ('p_valid_mc', 'p_exact', 'Monte Carlo resolution'),
                         ('p_old', 'p_valid_mc', 'old vs valid MC from the exact law (same B)')]:
        srows.append(dict(alpha=al, component=what, first=a_, second=b_, **paired(R_[a_], R_[b_])))
SQ = pd.DataFrame(srows); SQ.to_csv(OUT / '3c_separation.csv', index=False); display(SQ.round(4))
dev = QR.T_obs - MU0
print(f'old null mean minus exact mean: average {(QR.old_null_mean - MU0).mean():+.3f} (slope on T_obs - exact mean {np.polyfit(dev, QR.old_null_mean - MU0, 1)[0]:+.4f}); '
      f'without exclusion: average {(QR.old_noexcl_null_mean - MU0).mean():+.3f} (slope {np.polyfit(dev, QR.old_noexcl_null_mean - MU0, 1)[0]:+.4f})')

In [12]:
MAX_HOURS = 0   # overrides the section-4 run-time limit only (the SPEC is unchanged)

## 4. R row: the exact test on the surrogate series of notebook 32

I rebuild the surrogate scores with notebook 32's seeds, set Q_ν from the cell counts of each surrogate's alerts and compute the exact p; the resampler p comes from notebook 32's table (B = 400). I take the surrogates in a pre-set interleaved order (shift r, block r, …) and complete the first 214 of that order.

In [10]:
IDXE = np.flatnonzero(in_period(T, EVAL_START, EVAL_END)); s_eval = S0_FULL[IDXE]; n_eval = len(IDXE)
PARTS = np.array_split(np.arange(n_eval), N_SHUFFLE_BLOCKS)


def derangement(n, rng):
    while True:
        perm = rng.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm


def surrogate_score(family, r, rng):
    """Notebook 32 code: the surrogate M0 score on the full grid and a short description of the draw."""
    if family == 'shift':
        dmin = int(rng.integers(SHIFT_MIN_D * 1440, n_eval - SHIFT_MIN_D * 1440)) * int(rng.integers(0, 2) * 2 - 1)
        s_sh = np.full(len(T), np.nan); s_sh[IDXE] = np.roll(s_eval, dmin)
        return s_sh, dict(shift_minutes=dmin, blocks_moved=np.nan)
    perm = derangement(N_SHUFFLE_BLOCKS, rng)
    s_sh = np.full(len(T), np.nan); s_sh[IDXE] = np.concatenate([s_eval[PARTS[k]] for k in perm])
    return s_sh, dict(shift_minutes=np.nan, blocks_moved=int((perm != np.arange(N_SHUFFLE_BLOCKS)).sum()))


SUR = {}
for family, R_, seed_off in [('shift', R_SHIFT, 0), ('block_shuffle', R_BLOCK, 50000)]:
    rng = np.random.default_rng(SEED32 + 4 + seed_off)          # notebook 32 consumes this generator only in surrogate_score
    for r in range(R_):
        s_sh, info = surrogate_score(family, r, rng)
        ok_ = EVP & np.isfinite(s_sh); SUR[(family, r)] = emit_calendar_fast(T[ok_], s_sh[ok_] >= TH0)
ORDER = [x for r in range(R_BLOCK) for x in (('shift', r), ('block_shuffle', r))] + [('shift', r) for r in range(R_BLOCK, R_SHIFT)]
R_MEM_LIMIT_GIB = 6.0      # resource guard only (not an analysis setting, so not in the SPEC): largest single DP allocation allowed per surrogate
rpath = OUT / '4_r_row_pvalues.csv'
RR = pd.read_csv(rpath) if rpath.exists() else pd.DataFrame()
done = set(zip(RR.surrogate, RR.replication)) if len(RR) else set(); rows = RR.to_dict('records') if len(RR) else []
t_ = time.time(); n_new = 0; R_STOP_REASON = None
for pos, (family, r) in enumerate(ORDER):
    if (family, r) in done:
        continue
    if (time.time() - t_) / 3600 > MAX_HOURS:
        R_STOP_REASON = f'MAX_HOURS reached after {len(rows)} surrogates'
        print(R_STOP_REASON + '; the R row stops here'); break
    a = SUR[(family, r)]; ref = CP32[(CP32.surrogate == family) & (CP32.replication == r)].iloc[0]
    t_obs = fast_counts(a, EA4)['pre']
    if len(a) != int(ref.alerts) or t_obs != int(ref.pre_observed):
        raise RuntimeError(f'GUARD FAIL: regenerated surrogate {family} {r} differs from notebook 32')
    if t_additive(a) != t_obs:
        raise RuntimeError(f'GUARD FAIL: T is not additive for surrogate {family} {r}')
    tt0 = time.time(); P_, off_ = design_for(a)
    if off_:
        raise RuntimeError(f'Surrogate {family} {r} has {off_} alerts outside the Q candidate grid')
    _mem = (int(P_['R']) * (t_obs + 2) * int(P_['S'].max()) + int(P_['L']) * (t_obs + 2)) * 8 / 1024.0**3   # backward ring + stored full-state values
    if _mem > R_MEM_LIMIT_GIB:
        R_STOP_REASON = (f'STOP (resources): surrogate {family} {r} (order {pos}) needs about {_mem:.1f} GiB for the exact DP '
                         f'(max states/month {int(P_["S"].max()):,}, T_obs {t_obs}); limit {R_MEM_LIMIT_GIB} GiB')
        print(R_STOP_REASON + f'. {len(rows)} surrogates are saved; the report below uses the completed prefix of the fixed order.'); break
    law_, _ = exact_law(P_, t_obs + 1); rr = np.random.default_rng(SEED + 400000 + pos)
    pe, pr = p_from_law(law_, t_obs, rr.random())
    rows.append(dict(order=pos, surrogate=family, replication=r, alerts=len(a), alerts_off_grid=off_, pre_observed=t_obs, p_exact=pe, p_exact_randomized=pr,
                     p_old_nb32=float(ref.p_budget_preserving), old_null_mean_nb32=float(ref.budget_null_mean), state_size_max=int(P_['S'].max()), seconds=time.time() - tt0))
    n_new += 1
    if n_new % 5 == 0 or pos == len(ORDER) - 1:
        pd.DataFrame(rows).to_csv(rpath, index=False)
        el = time.time() - t_; print(f'R row {len(rows)}/{len(ORDER)} ({el / 60:.1f} min; about {el / n_new * (len(ORDER) - len(rows)) / 60:.0f} min left)', flush=True)
RR = pd.DataFrame(rows).sort_values('order').reset_index(drop=True) if rows else pd.DataFrame(columns=['order', 'surrogate', 'replication', 'p_exact', 'p_exact_randomized', 'p_old_nb32'])
if len(RR):
    RR.to_csv(rpath, index=False)
# report the completed prefix of the fixed order only
pref = int(np.argmin(np.r_[np.diff(RR.order.to_numpy()) == 1, False])) + 1 if len(RR) and RR.order.iloc[0] == 0 else 0
RRp = RR.iloc[:pref]
print(f'R row: {len(RRp)} surrogates in the fixed order are complete' + ('' if len(RRp) >= R_MIN_REPORT else f' (fewer than {R_MIN_REPORT}: report as incomplete)'))
rrows = []
for fam in ['shift', 'block_shuffle', 'both']:
    sub = RRp if fam == 'both' else RRp[RRp.surrogate == fam]
    if not len(sub):
        continue
    for al in ALPHAS:
        xo = (sub.p_old_nb32 <= al).to_numpy(); xe = (sub.p_exact <= al).to_numpy(); xr = (sub.p_exact_randomized <= al).to_numpy()
        for lab, x in [('old draw_budget (notebook 32, B=400)', xo), ('exact (no MC error)', xe), ('randomized exact', xr)]:
            lo, hi, _, _ = rate_ci(x.sum(), len(x))
            rrows.append(dict(surrogates=fam, alpha=al, test=lab, n=len(x), rejections=int(x.sum()), rate=x.mean(), lo95=lo, hi95=hi, vs_nominal=verdict(lo, hi, al)))
        rrows.append(dict(surrogates=fam, alpha=al, test='paired: old - exact', n=len(xo), **paired(xo, xe)))
RRR = pd.DataFrame(rrows); RRR.to_csv(OUT / '4b_r_row_rates.csv', index=False); display(RRR.round(4))

R row 130/600 (23.0 min; about 2158 min left)
R row 135/600 (43.2 min; about 2009 min left)
R row 140/600 (66.0 min; about 2024 min left)
R row 145/600 (93.4 min; about 2124 min left)
R row 150/600 (120.5 min; about 2170 min left)
R row 155/600 (143.7 min; about 2132 min left)
R row 160/600 (168.6 min; about 2120 min left)
R row 165/600 (195.7 min; about 2128 min left)
R row 170/600 (217.3 min; about 2076 min left)
R row 175/600 (238.7 min; about 2029 min left)
R row 180/600 (261.6 min; about 1998 min left)
R row 185/600 (285.0 min; about 1971 min left)
R row 190/600 (304.9 min; about 1923 min left)
R row 195/600 (329.3 min; about 1905 min left)
R row 200/600 (351.8 min; about 1876 min left)
R row 205/600 (376.6 min; about 1859 min left)
R row 210/600 (400.5 min; about 1838 min left)
MAX_HOURS reached after 214 surrogates; the R row stops here
R row: 214 surrogates in the fixed order are complete


,surrogates,alpha,test,n,rejections,rate,lo95,hi95,vs_nominal,diff,diff_lo,diff_hi,only_first,only_second,mcnemar_p
0,shift,0.01,"old draw_budget (notebook 32, B=400)",107,2.0,0.0187,0.0023,0.0659,no evidence of size distortion,NaN,NaN,NaN,NaN,NaN,NaN
1,shift,0.01,exact (no MC error),107,1.0,0.0093,0.0002,0.0510,no evidence of size distortion,NaN,NaN,NaN,NaN,NaN,NaN
2,shift,0.01,randomized exact,107,1.0,0.0093,0.0002,0.0510,no evidence of size distortion,NaN,NaN,NaN,NaN,NaN,NaN
3,shift,0.01,paired: old - exact,107,NaN,NaN,NaN,NaN,NaN,0.0093,-0.0089,0.0276,1.0,0.0,1.0
4,shift,0.05,"old draw_budget (notebook 32, B=400)",107,6.0,0.0561,0.0209,0.1181,no evidence of size distortion,NaN,NaN,NaN,NaN,NaN,NaN
5,shift,0.05,exact (no MC error),107,6.0,0.0561,0.0209,0.1181,no evidence of size distortion,NaN,NaN,NaN,NaN,NaN,NaN
6,shift,0.05,randomized exact,107,9.0,0.0841,0.0392,0.1537,no evidence of size distortion,NaN,NaN,NaN,NaN,NaN,NaN
7,shift,0.05,paired: old - exact,107,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0,0.0,1.0
8,shift,0.10,"old draw_budget (notebook 32, B=400)",107,9.0,0.0841,0.0392,0.1537,no evidence of size distortion,NaN,NaN,NaN,NaN,NaN,NaN
9,shift,0.10,exact (no MC error),107,9.0,0.0841,0.0392,0.1537,no evidence of size distortion,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Summary (the pre-specified tables)

In [11]:
def pick(df, **kw):
    s = df
    for k, v in kw.items():
        s = s[s[k] == v]
    return s.to_dict('records')


summary = dict(spec_sha256=SPEC_SHA, run_time=str(pd.Timestamp.now(tz='UTC').floor('s')), status=SPEC['status'],
               verification=dict(small_designs=int(len(VR)), max_law_error=float(VR.law_err.max()), sampler_designs=n_samp, real_design_chi_square_p=chi_real,
                                 real_design_violations=int(bs + bc), implementation_check='PASS' if impl_ok else 'FAIL'),
               m0=T2.iloc[0].to_dict(), q_row_rates=RQ.to_dict('records'), q_row_separation=SQ.to_dict('records'),
               r_row=dict(completed_in_order=int(len(RRp)), target=len(ORDER), complete=bool(len(RRp) == len(ORDER)), stop_reason=R_STOP_REASON, rates=RRR.to_dict('records')))
(OUT / 'EXACT_NULL_SUMMARY.json').write_text(json.dumps(summary, indent=2, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else (bool(o) if isinstance(o, np.bool_) else str(o))))
print('Done. Outputs in', OUT)

Done. Outputs in /content/drive/MyDrive/btc_lead_cost_results_77months_spot/exact_conditional_null_v1
